побудуємо повноцінне ML-рішення реальної ML-задачі.
## Dataset

Будемо працювати з набором даних `cars.csv`, який описує автомобілі і їх ціну у індійських рупіях :) Мета - передбачити ціну авто за його характеристиками. Опис набору даних:

| Назва рядка            | Опис                                                                                                                                                 |
|------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------|
| Car_ID                 | Унікальний ідентифікатор для кожного оголошення про автомобіль.                                                                                         |
| Brand                  | Бренд або виробник автомобіля (наприклад, Toyota, Honda, Ford тощо).                                                                                     |
| Model                  | Модель автомобіля (наприклад, Camry, Civic, Mustang тощо).                                                                                               |
| Year                   | Рік виготовлення автомобіля.                                                                                                                            |
| Kilometers_Driven      | Загальний пробіг автомобіля у кілометрах.                                                                                                               |
| Fuel_Type              | Тип палива, який використовує автомобіль (наприклад, бензин, дизель, електро тощо).                                                                      |
| Transmission           | Тип трансмісії автомобіля (наприклад, механічна, автоматична).                                                                                           |
| Owner_Type             | Кількість попередніх власників автомобіля (наприклад, перший, другий, третій).                                                                           |
| Mileage                | Паливна ефективність автомобіля у кілометрах на літр.                                                                                                   |
| Engine                 | Об'єм двигуна автомобіля в кубічних сантиметрах (CC).                                                                                                   |
| Power                  | Максимальна потужність автомобіля в кінських силах (bhp).                                                                                               |
| Seats                  | Кількість місць в автомобілі.                                                                                                                           |
| Price                  | Вартість автомобіля в INR (індійські рупії), що є цільовою змінною для прогнозування.                                                                   |

# Імпорти

Для зручності рекомендую всі імпорти розмістити тут нагорі, аби коли ви перезавантажували ноутбук, одразу можна було в один запуск клітинки імпортувати всі потрібні бібліотеки.

In [160]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sympy.abc import theta
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# Знайомство з даними

**Завдання 1.** Завантажте набір даних `cars.csv` в pandas.DataFrame. Виведіть перші 5 записів.

In [98]:
cars_df = pd.read_csv('/Users/macbook/Desktop/machine_learning_course/cars.csv')
cars_df.head()

,Car_ID,Brand,Model,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price
0,1,Toyota,Corolla,2018,50000,Petrol,Manual,First,15,1498,108,5,800000
1,2,Honda,Civic,2019,40000,Petrol,Automatic,Second,17,1597,140,5,1000000
2,3,Ford,Mustang,2017,20000,Petrol,Automatic,First,10,4951,395,4,2500000
3,4,Maruti,Swift,2020,30000,Diesel,Manual,Third,23,1248,74,5,600000
4,5,Hyundai,Sonata,2016,60000,Diesel,Automatic,Second,18,1999,194,5,850000


**Завдання 2.** Виведіть типи даних колонок даних, а також дослідіть, які по факту типи даних мають записи в кожній колонці (тип `object` може містити різні типи даних) і скільки значень є в кожній категоріальній колонці.

Напишіть висновок, скільки в наборі даних числових та категоріальних колонок кожного з трьох різних типів (бінарна, мільтикатегоріальна без порядку, мультикатегоріальна з порядком). Шаблон висновку

```
В наборі даних 10 числових і 10 категоріальних колонок з них
- 2 бінарні (мають лише 2 значення)
- 6 мультикатегоріальних (більше 2х значень) зі значеннями, для яких немає відношення порядку
- 2 колонки, в яких можна встановити відношення порядку (наприклад Small<Medium<Large)
```

Якщо не знаєте, як це зробити з `pandas` - ось підказка, які методи можуть допомогти вам виконати це завдання

- pandas.DataFrame.info()
- pandas.DataFrame.dtypes
- pandas.DataFrame.loc[...]
- pandas.DataFrame.select_dtypes(...)
- pandas.Series.unique()
- pandas.Series.nunique()

Детальніше ознайомитись з кожним ви можете в [документації](https://pandas.pydata.org/docs/reference/frame.html), або написати в окремій клітинці знак питання і назву методу (тільки приберіть це перед здачею, бо перегляд документації - не допомагає зрозуміти дані і хід думок, а Ваша робота - це як презентація замовнику зробленої задачі).


In [99]:
cars_df.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Car_ID             100 non-null    int64 
 1   Brand              100 non-null    object
 2   Model              100 non-null    object
 3   Year               100 non-null    int64 
 4   Kilometers_Driven  100 non-null    int64 
 5   Fuel_Type          100 non-null    object
 6   Transmission       100 non-null    object
 7   Owner_Type         100 non-null    object
 8   Mileage            100 non-null    int64 
 9   Engine             100 non-null    int64 
 10  Power              100 non-null    int64 
 11  Seats              100 non-null    int64 
 12  Price              100 non-null    int64 
dtypes: int64(8), object(5)
memory usage: 10.3+ KB


In [100]:
# statistics
cars_df.describe()

,Car_ID,Year,Kilometers_Driven,Mileage,Engine,Power,Seats,Price
count,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,100.000000,1.000000e+02
mean,50.500000,2018.39000,28150.000000,17.210000,1855.230000,158.130000,5.230000,1.574000e+06
std,29.011492,1.17116,9121.375716,3.309902,631.311475,76.968137,0.750151,1.000265e+06
min,1.000000,2016.00000,10000.000000,10.000000,999.000000,68.000000,4.000000,4.500000e+05
25%,25.750000,2017.75000,22000.000000,15.000000,1462.000000,103.000000,5.000000,7.000000e+05
50%,50.500000,2018.00000,27000.000000,17.000000,1774.000000,148.000000,5.000000,1.300000e+06
75%,75.250000,2019.00000,32000.000000,19.000000,2143.000000,187.000000,5.000000,2.500000e+06
max,100.000000,2021.00000,60000.000000,25.000000,4951.000000,396.000000,7.000000,4.000000e+06


In [101]:
# categorical variables
cars_df.select_dtypes(include = "object").columns

Index(['Brand', 'Model', 'Fuel_Type', 'Transmission', 'Owner_Type'], dtype='object')

In [102]:
cars_df.select_dtypes(include = "object").nunique()

Brand           11
Model           58
Fuel_Type        2
Transmission     2
Owner_Type       3
dtype: int64

In [103]:
# categ. variables amount
len(cars_df.select_dtypes(include = "object").columns)

5

In [104]:
# numeric variables
cars_df.select_dtypes(include="number")


,Car_ID,Year,Kilometers_Driven,Mileage,Engine,Power,Seats,Price
0,1,2018,50000,15,1498,108,5,800000
1,2,2019,40000,17,1597,140,5,1000000
2,3,2017,20000,10,4951,395,4,2500000
3,4,2020,30000,23,1248,74,5,600000
4,5,2016,60000,18,1999,194,5,850000
...,...,...,...,...,...,...,...,...
95,96,2019,22000,16,1950,191,5,2900000
96,97,2017,38000,13,2755,171,7,1400000
97,98,2018,26000,18,1497,121,5,750000
98,99,2019,24000,17,1497,113,5,850000


In [105]:
len(cars_df.select_dtypes(include = "number").columns)

8

Висновок: У наборі даних "Cars" 13 колонок:
- числових - 8
- категоріальних - 5
Категоріальні колонки діляться на:
- бінарні - 2 (Fuel_Type, Transmission)
- мультикатегоріальні - 2 (Brand, Model)
- мультикатегоріальна з порядком - 1 (Owner_Type: first, second, third)

**Завдання 3**. Розділіть дані на тренувальні і тест. Відведіть в тест 20%, поставте `random_state=12`. Ми будемо передбачати колонку `Price` - тож, вона є цільовою змінною. В результаті у Вас має бути 4 набори даних `X_train, X_test, y_train, y_test`.

Надалі ми всюди тренуємо методи для кодування, масштабування та саму модель тільки на тренувальних даних X_train (та y_train для моделі), а на тестувальних лише використовуємо вже навчені методи для кодування, масштабування і модель викликаючи в них `transform()` (для методів обробки даних) або `predict()` (для моделі).

І так само треба робити завжди.

In [106]:
# Features and target
X = cars_df.drop(columns = "Price")
y = cars_df["Price"]

# train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 12)

**Завдання 4**. Кодуємо категоріальні колонки.

1. Закодуйте колонки з бінарними значеннями `Fuel_Type` і `Transmission` так, аби вони були у чисельному представленні і містили лише 0 так 1. Приклад був у лекції. Значення 1 нехай буде у категорії, яка містить більше значень в колонці.

2. Закодуйте колонку `Brand` з використанням `sklearn.preprocessing.OneHotEncoder` аналогічно до того, як ми робили це в лекції. Увага! Ми робимо виклик методу `Encoder.fit()` на тренувальних даних `X_train`, а на тестувальних тільки викликаємо `Encoder.transform()`. Додайте закодовані значення в набори даних `X_train`, `X_test`.

3. Колонку `Owner_Type` з використанням `sklearn.preprocessing.OrdinalEncoder` я закодую для вас. Проаналізуйте, що відбувається.

Колонка `Model` містять забагато значень для кодування в тому вигляді, як вона є зараз, з огляду на невелику кількість даних. Якщо ви бачите, як можна згрупувати значення в цій колонці скоротивши кількість унікальних значень до 3-5, то можете виконати ще цю трансформацію, використати цю колонку в моделі і отримати додаткову практику і бал, якщо все буде зроблено правильно. А якщо неправильно - то фідбек на Вашу роботу :)

In [109]:
# binary encoding
X_train['Fuel_Type'] = X_train['Fuel_Type'].astype(str).str.strip()
X_test['Fuel_Type'] = X_test['Fuel_Type'].astype(str).str.strip()

X_train['Transmission'] = X_train['Transmission'].astype(str).str.strip()
X_test['Transmission'] = X_test['Transmission'].astype(str).str.strip()

fuel_codes = {'Petrol': 1, 'Diesel': 0}
transmission_codes = {'Automatic': 1, 'Manual': 0}

X_train['Fuel_Type'] = X_train['Fuel_Type'].map(fuel_codes)
X_test['Fuel_Type'] = X_test['Fuel_Type'].map(fuel_codes)

X_train['Transmission'] = X_train['Transmission'].map(transmission_codes)
X_test['Transmission'] = X_test['Transmission'].map(transmission_codes)

In [110]:
# OneHotEncoder
'Brand' in X_train.columns

True

In [111]:
enc = preprocessing.OneHotEncoder(handle_unknown = "ignore")

In [112]:
enc.fit(X_train[['Brand']])

OneHotEncoder(handle_unknown='ignore')

In [113]:
brand_one_train = enc.transform(X_train[['Brand']]).toarray()
brand_one_test = enc.transform(X_test[['Brand']]).toarray()

In [114]:
brand_cols = enc.categories_[0]
brand_one_train_df = pd.DataFrame(data = brand_one_train, columns = brand_cols, index = X_train.index)
brand_one_test_df = pd.DataFrame(data = brand_one_test, columns = brand_cols, index = X_test.index)

In [115]:
# delete Brand, add one-hot columns
X_train = X_train.drop(columns = 'Brand')
X_test = X_test.drop(columns = 'Brand')

X_train = pd.concat([X_train, brand_one_train_df], axis = 1)
X_test = pd.concat([X_test, brand_one_test_df], axis = 1)

In [116]:
# check changes
'Brand' in X_train.columns



False

In [117]:
'Brand' in X_test.columns

False

In [81]:
#train_only = set(X_train.columns) - set(X_test.columns)
#test_only = set(X_test.columns) - set(X_train.columns)
#print(train_only, test_only)

set() {'Owner_Type', 'Model'}


In [118]:
X_train.shape, X_test.shape
X_train.head()

,Car_ID,Model,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Audi,BMW,Ford,Honda,Hyundai,Mahindra,Maruti,Mercedes,Tata,Toyota,Volkswagen
83,84,T-Roc,2019,22000,NaN,NaN,Second,18,1498,148,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
55,56,A5,2018,28000,NaN,NaN,First,17,1968,187,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26,27,A6,2018,28000,NaN,NaN,First,15,1984,241,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
54,55,Vento,2017,32000,NaN,NaN,Second,18,1598,103,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
19,20,GLC,2017,26000,NaN,NaN,Second,12,1991,241,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [119]:
X_train['Fuel_Type'].value_counts()

Series([], Name: count, dtype: int64)

In [120]:
X_train['Transmission'].value_counts()

Series([], Name: count, dtype: int64)

In [121]:
ordenc = OrdinalEncoder(categories=[['First', 'Second', 'Third']]) # визначаємо порядок категорій
ordenc.fit(X_train[['Owner_Type']])

X_train['Owner_Type_Codes'] = ordenc.transform(X_train[['Owner_Type']])
X_test['Owner_Type_Codes'] = ordenc.transform(X_test[['Owner_Type']])

Очікуваний результат після трансформацій:

In [122]:
pd.set_option('display.max_columns', 100)
display(X_train.head()), display(X_test.head())

,Car_ID,Model,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Audi,BMW,Ford,Honda,Hyundai,Mahindra,Maruti,Mercedes,Tata,Toyota,Volkswagen,Owner_Type_Codes
83,84,T-Roc,2019,22000,NaN,NaN,Second,18,1498,148,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
55,56,A5,2018,28000,NaN,NaN,First,17,1968,187,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26,27,A6,2018,28000,NaN,NaN,First,15,1984,241,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
54,55,Vento,2017,32000,NaN,NaN,Second,18,1598,103,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
19,20,GLC,2017,26000,NaN,NaN,Second,12,1991,241,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


,Car_ID,Model,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Audi,BMW,Ford,Honda,Hyundai,Mahindra,Maruti,Mercedes,Tata,Toyota,Volkswagen,Owner_Type_Codes
17,18,Q3,2016,38000,NaN,NaN,Second,15,1395,148,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
41,42,Santro,2019,26000,NaN,NaN,Third,20,1086,68,5,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
92,93,Vento,2017,32000,NaN,NaN,Second,18,1598,103,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
14,15,Ertiga,2020,18000,NaN,NaN,First,19,1462,103,7,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
68,69,Aspire,2019,26000,NaN,NaN,Third,20,1194,94,5,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0


(None, None)

Висновок: шматок коду вище визначає порядок категорій. З'явилась колонка 'Owner_Type_Codes' в X_train та X_test

**Завдання 5.** Оберіть лише числові колонки з `X_train` (можна для цього використати `pandas.select_dtypes(...)` або видалити всі НЕчислові дані, вони нам вже не потрібні), обʼєднайте ці дані з `y_train` (зручно з `pandas.concat([df1, df2], axis=1)`), побудуйте для цих даних матрицю кореляції і проаналізуйте її. Напишіть висновок, які колонки корелюють з цільовою змінною на більш ніж 0.5 за модулем (може бути як позитивна, так і негативна кореляція).

In [123]:
# num_cols only
X_train_num = X_train.select_dtypes(include = "number")
X_train_num.dtypes


Car_ID                 int64
Year                   int64
Kilometers_Driven      int64
Fuel_Type            float64
Transmission         float64
Mileage                int64
Engine                 int64
Power                  int64
Seats                  int64
Audi                 float64
BMW                  float64
Ford                 float64
Honda                float64
Hyundai              float64
Mahindra             float64
Maruti               float64
Mercedes             float64
Tata                 float64
Toyota               float64
Volkswagen           float64
Owner_Type_Codes     float64
dtype: object

In [125]:
# add target Price
train_corr_df = pd.concat([X_train_num, pd.Series(y_train, name = 'Price')], axis = 1)
corr_matrix = train_corr_df.corr(numeric_only=True)
corr_price = corr_matrix['Price'].sort_values(ascending = False)
strong_corr = corr_price[abs(corr_price) > 0.5].sort_values(ascending = False)
print(strong_corr)

Price      1.000000
Power      0.849137
Engine     0.710561
Mileage   -0.638404
Name: Price, dtype: float64


Висновок: обчислення кореляції з нашим таргетом 'Price', показало, що з ціною дуже корелюють ознаки 'Power', 'Engine', 'Mileage'. Ці характеристики сильно впливають на ціну авто. 'Mileage' від'ємний -- має від'ємну кореляцію з вартістю авто. Чим вище пробіг, тим дешевше авто. Це логічно і підтверждує коректкність обчислень.

**Завдання 6**. Тренуємо лінійну регресію.
0. Видаліть усі НЕчислові колонки з `X_train`, `X_test`, якщо ще цього не зробили.
1. Натренуйте лінійну регресую з `sklearn` на усіх числових даних тренувального набору `X_train`.
2. Зробіть передбачення на  `X_train`, `X_test`. Знайдіть і виведіть root mean squared error відхилення передбачення від справжніх значень цільової змінної.
3. Побудуйте графік розсіювання передбачень проти реальних даних цільової змінної для тренувального і тестувального наборів даних. Що можете сказати про якість моделі?

In [143]:
X_train.dtypes

Car_ID                 int64
Year                   int64
Kilometers_Driven      int64
Mileage                int64
Engine                 int64
Power                  int64
Seats                  int64
Audi                 float64
BMW                  float64
Ford                 float64
Honda                float64
Hyundai              float64
Mahindra             float64
Maruti               float64
Mercedes             float64
Tata                 float64
Toyota               float64
Volkswagen           float64
Owner_Type_Codes     float64
dtype: object

In [144]:
# drop non_numeric columns
non_num_train = set(X_train.select_dtypes(exclude = "number").columns)
non_num_test = set(X_test.select_dtypes(exclude = "number").columns)
non_numeric_cols = list(non_num_train | non_num_test)

X_train = X_train.drop(columns = non_numeric_cols, errors = 'ignore')
X_test = X_test.drop(columns = non_numeric_cols, errors = 'ignore')
print('non-num left in train:', X_train.select_dtypes(exclude = "number").columns.tolist())
print(X_train.columns.equals(X_test.columns))




non-num left in train: []
True


In [145]:
X_train.isna().sum().sort_values(ascending = False)

Car_ID               0
Year                 0
Kilometers_Driven    0
Mileage              0
Engine               0
Power                0
Seats                0
Audi                 0
BMW                  0
Ford                 0
Honda                0
Hyundai              0
Mahindra             0
Maruti               0
Mercedes             0
Tata                 0
Toyota               0
Volkswagen           0
Owner_Type_Codes     0
dtype: int64

In [146]:
# NaN
X_train = X_train.dropna(axis = 1, how = "all")
X_test = X_test.dropna(axis = 1, how = "all")
X_train.isna().sum()

Car_ID               0
Year                 0
Kilometers_Driven    0
Mileage              0
Engine               0
Power                0
Seats                0
Audi                 0
BMW                  0
Ford                 0
Honda                0
Hyundai              0
Mahindra             0
Maruti               0
Mercedes             0
Tata                 0
Toyota               0
Volkswagen           0
Owner_Type_Codes     0
dtype: int64

In [147]:
# Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)


LinearRegression()

In [148]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

**Завдання 7**. Ми хочемо проаналізувати як впливає кожен чинник на цілову змінну. Для цього давайте промасштабуємо наші дані.
1. Зробіть масштабування незалежних змінних використовуючи `StandardScaler`. Тренуємо на тренувальних даних, а на тестувальних лише викликаємо `transform`.

2. Натренуйте модель на відмасштабованих даних і перегляньте коефіцієнти моделі. Які колонки є найвпливовішими на формування передбачення з точки зору коефіцієнтів? Проаналізуйте напрям дії найважливіших коефіцієнтів. Чи це логічно з точки зору значення відповідних змінних, що вони впливають на цільову змінну саме в напрямі збільшення / зменшення?

In [165]:
scaler = StandardScaler()

# scaler for train only
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns = X_train.columns, index = X_train.index)

# transform for test only
X_test_scaled = scaler.transform(X_test)

In [166]:
# model scaled
model_scaled = LinearRegression()
model_scaled.fit(X_train_scaled, y_train)



LinearRegression()

In [167]:
feature_names = X_train.columns
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': model_scaled.coef_})

coef_df.sort_values('coefficient', ascending = False)
coef_df

,feature,coefficient
0,Car_ID,10391.901257
1,Year,29646.157096
2,Kilometers_Driven,6270.745117
3,Mileage,-28338.022658
4,Engine,94812.492973
5,Power,460238.563128
6,Seats,77048.459247
7,Audi,219326.973011
8,BMW,263621.151556
9,Ford,-99983.523443


Висновок: У результаті масштабування коеф. лінійної регресії видно наступне:
1. На збільшення вартості впливають: 'Year', 'Engine', 'Power', преміальність брендів: BMW, Audi, Mercedes. Тому тут логічні показники.
2. На зниження вартості впливають: 'Mileage', середній клас брендів: Ford, Tata тощо. 'Owner_Type_Codes': від'ємний показник - чим більше власників, тим нижче вартість. Також усе логічно.

**Завдання 8.** На тих самих відмасштабованих даних натренуйте модель з `statsmodels`. Виведіть звіт і проаналізуйте p-value коефіцієнтів. Які ознаки є стат значущими на рівні значущості 0.05? Напишіть їх список.

In [168]:
X_train_sm = sm.add_constant(X_train_scaled)

In [169]:
model_sm = sm.OLS(y_train, X_train_sm)
result = model_sm.fit()

In [170]:
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  Price   R-squared:                       0.940
Model:                            OLS   Adj. R-squared:                  0.923
Method:                 Least Squares   F-statistic:                     53.36
Date:                Wed, 28 Jan 2026   Prob (F-statistic):           1.60e-30
Time:                        16:57:03   Log-Likelihood:                -1108.7
No. Observations:                  80   AIC:                             2255.
Df Residuals:                      61   BIC:                             2301.
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const               1.68e+06   3.23e+04     51.933      0.000    1.62e+06    1.74e+06
Car_ID             1.039e+04   3.85e+04      0.270      0.788   -6.65e+04    8.73e+04
Year               2.965e+04   7.11e+04      0.417      0.678   -1.12e+05    1.72e+05
Kilometers_Driven  6270.7451   6.99e+04      0.090      0.929   -1.34e+05    1.46e+05
Mileage           -2.834e+04   6.35e+04     -0.446      0.657   -1.55e+05    9.86e+04
Engine             9.481e+04   7.94e+04      1.193      0.237   -6.41e+04    2.54e+05
Power              4.602e+05   7.86e+04      5.859      0.000    3.03e+05    6.17e+05
Seats              7.705e+04   4.53e+04      1.699      0.094   -1.36e+04    1.68e+05
Audi               2.193e+05   3.45e+04      6.360      0.000     1.5e+05    2.88e+05
BMW                2.636e+05   3.52e+04      7.493      0.000    1.93e+05    3.34e+05
Ford              -9.998e+04   3.61e+04     -2.769      0.007   -1.72e+05   -2.78e+04
Honda             -1.174e+05   3.86e+04     -3.042      0.003   -1.95e+05   -4.02e+04
Hyundai           -1.343e+05   3.39e+04     -3.966      0.000   -2.02e+05   -6.66e+04
Mahindra          -1.234e+05   3.33e+04     -3.706      0.000    -1.9e+05   -5.68e+04
Maruti            -1.156e+05   3.44e+04     -3.362      0.001   -1.84e+05   -4.68e+04
Mercedes           2.671e+05   3.47e+04      7.698      0.000    1.98e+05    3.37e+05
Tata              -1.646e+05   4.05e+04     -4.060      0.000   -2.46e+05   -8.35e+04
Toyota            -4.989e+04   3.69e+04     -1.353      0.181   -1.24e+05    2.38e+04
Volkswagen        -4.901e+04   3.36e+04     -1.459      0.150   -1.16e+05    1.81e+04
Owner_Type_Codes  -4.946e+04   4.85e+04     -1.020      0.312   -1.46e+05    4.75e+04
==============================================================================
Omnibus:                        0.059   Durbin-Watson:                   1.962
Prob(Omnibus):                  0.971   Jarque-Bera (JB):                0.171
Skew:                           0.059   Prob(JB):                        0.918
Kurtosis:                       2.807   Cond. No.                     7.06e+15
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 5.49e-30. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [171]:
p_values = result.pvalues
significant_features = p_values[p_values < 0.05]
significant_features

const       3.361375e-52
Power       2.018021e-07
Audi        2.876019e-08
BMW         3.283650e-10
Ford        7.439556e-03
Honda       3.465284e-03
Hyundai     1.943545e-04
Mahindra    4.557489e-04
Maruti      1.340028e-03
Mercedes    1.454218e-10
Tata        1.423373e-04
dtype: float64

Висновок: на рівні стат.значущості 0.05 перевага у 'Power' та брендів. А рік, пробіг, двигун у моєму результаті не мають стат.значущості. Не маю відповіді, чому. Вище у приміткам мова про проблеми з мультиколінеарністю матриці.

**Завдання 9**. Натренуйте лінійну регресію з `statsmodels` тільки на ознаках, які виявлись стат. значущими в попередньому завданні. Проаналізуйте показники моделі. Чи значно змінились R2 і Adj. R-squared?

In [172]:
significant_features_names = significant_features.index.drop('const')

In [173]:
X_train_sig = X_train_scaled[significant_features_names]

In [174]:
# intercept
X_train_sig = sm.add_constant(X_train_sig)

In [175]:
model_sm_sig = sm.OLS(y_train, X_train_sig)
result_sig = model_sm_sig.fit()

In [176]:
result_sig.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  Price   R-squared:                       0.927
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     87.23
Date:                Wed, 28 Jan 2026   Prob (F-statistic):           4.08e-35
Time:                        17:25:12   Log-Likelihood:                -1116.9
No. Observations:                  80   AIC:                             2256.
Df Residuals:                      69   BIC:                             2282.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        1.68e+06   3.37e+04     49.854      0.000    1.61e+06    1.75e+06
Power        5.63e+05   4.72e+04     11.920      0.000    4.69e+05    6.57e+05
Audi        2.637e+05    4.1e+04      6.428      0.000    1.82e+05    3.46e+05
BMW         2.912e+05   4.48e+04      6.497      0.000    2.02e+05    3.81e+05
Ford       -8.511e+04   4.27e+04     -1.995      0.050    -1.7e+05     -10.780
Honda      -7.202e+04    3.8e+04     -1.894      0.062   -1.48e+05    3856.184
Hyundai    -1.298e+05   3.85e+04     -3.367      0.001   -2.07e+05   -5.29e+04
Mahindra   -8.524e+04    3.7e+04     -2.301      0.024   -1.59e+05   -1.14e+04
Maruti     -9.535e+04   3.82e+04     -2.494      0.015   -1.72e+05   -1.91e+04
Mercedes     2.95e+05   4.33e+04      6.820      0.000    2.09e+05    3.81e+05
Tata        -1.12e+05   4.16e+04     -2.692      0.009   -1.95e+05    -2.9e+04
==============================================================================
Omnibus:                       12.066   Durbin-Watson:                   1.865
Prob(Omnibus):                  0.002   Jarque-Bera (JB):               14.138
Skew:                           0.738   Prob(JB):                     0.000851
Kurtosis:                       4.436   Cond. No.                         3.05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Висновок: 1. У попередній моделі були наступні показники: R-sq. 0.940 (94%) - це скільки модель змогла загалом пояснити та Adj R-sq. 0.923 (92%) - скільки пояснила "чесно". Різниця 2% вказує на те, що був якийсь шум, який вплинув на результат моделі.
2. У поточній моделі наступні результати: R-sq. 0.927 (92%)та R-sq. 0.916 (91%), різниця приблизно 1%, що покращує результат роботи моделі. Це ще неідеально, але краще.

**Завдання 10**. Натренуйте лінійну регресію з `statsmodels` на усіх ознаках з масштабованого `X_train`, у яких p_value в завданні 8 менше за `0.25`. Ми таким чином помʼякшили критерій відбору ознак. Проаналізуйте показники моделі. Чи значно змінились R2 і Adj. R-squared порівняно з завданням 8? Яку модель з останніх 3х завдань ви б лишили для використання?

In [177]:
features_025 = p_values[p_values < 0.25].index.drop('const')

In [178]:
X_train_025 = X_train_scaled[features_025]
X_train_025 = sm.add_constant(X_train_025)
model_sm_025 = sm.OLS(y_train, X_train_025)
result_025 = model_sm_025.fit()

In [179]:
result_025.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  Price   R-squared:                       0.938
Model:                            OLS   Adj. R-squared:                  0.926
Method:                 Least Squares   F-statistic:                     77.35
Date:                Wed, 28 Jan 2026   Prob (F-statistic):           1.05e-34
Time:                        17:59:21   Log-Likelihood:                -1109.9
No. Observations:                  80   AIC:                             2248.
Df Residuals:                      66   BIC:                             2281.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        1.68e+06   3.16e+04     53.191      0.000    1.62e+06    1.74e+06
Engine      9.654e+04   6.59e+04      1.465      0.148    -3.5e+04    2.28e+05
Power       4.848e+05   7.09e+04      6.838      0.000    3.43e+05    6.26e+05
Seats        7.47e+04   3.93e+04      1.902      0.061   -3698.215    1.53e+05
Audi        2.236e+05   2.98e+04      7.491      0.000    1.64e+05    2.83e+05
BMW         2.623e+05   3.18e+04      8.237      0.000    1.99e+05    3.26e+05
Ford       -1.109e+05      3e+04     -3.691      0.000   -1.71e+05   -5.09e+04
Honda      -1.147e+05   3.13e+04     -3.669      0.000   -1.77e+05   -5.23e+04
Hyundai     -1.51e+05   3.02e+04     -5.001      0.000   -2.11e+05   -9.07e+04
Mahindra    -1.18e+05   3.07e+04     -3.851      0.000   -1.79e+05   -5.68e+04
Maruti     -1.172e+05   3.11e+04     -3.766      0.000   -1.79e+05   -5.51e+04
Mercedes    2.702e+05   3.05e+04      8.870      0.000    2.09e+05    3.31e+05
Tata       -1.432e+05   3.07e+04     -4.671      0.000   -2.04e+05    -8.2e+04
Toyota     -4.649e+04   3.25e+04     -1.430      0.157   -1.11e+05    1.84e+04
Volkswagen -6.144e+04   2.96e+04     -2.073      0.042   -1.21e+05   -2251.089
==============================================================================
Omnibus:                        0.256   Durbin-Watson:                   1.950
Prob(Omnibus):                  0.880   Jarque-Bera (JB):                0.420
Skew:                           0.102   Prob(JB):                        0.811
Kurtosis:                       2.709   Cond. No.                     4.56e+15
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 9.55e-30. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

Висновок: У третій моделі наступні показники: R-sq. = 0.938 (93%) Adj.R-sq. = 0.926 (92%) при значенні            p_value < 0.25. Після пом'якшення критерію p_value < 0.25, модель покращила R-sq. та Adj.R-sq. Отримано хороший результат. Це найбільш оптимальний варіант для роботи в порівнянні з попередніми двома моделями. Залишила би саме цю.